# Sports Scraper — EDA
Exploratory analysis of scraped posts stored in MongoDB.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pymongo', 'wordcloud', 'seaborn', 'pandas', 'matplotlib'])
print('Done')

: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['sports_scraper']
collection = db['posts']

df = pd.DataFrame(list(collection.find()))
if df.empty:
    print('No data in DB. Run a search and save first!')
else:
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print(f'Total posts: {len(df)}')
    print(df['source'].value_counts().to_dict())
    df.head()

## Posts by Source

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart
    source_counts = df['source'].value_counts()
    colors = ['#6C63FF', '#48CAE4']
    source_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='none')
    axes[0].set_title('Posts by Source', fontsize=14)
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=0)

    # Pie chart
    axes[1].pie(source_counts, labels=source_counts.index, autopct='%1.0f%%',
                colors=colors, startangle=90)
    axes[1].set_title('Source Distribution', fontsize=14)

    plt.tight_layout()
    plt.show()

## Sentiment Analysis

In [ ]:
if not df.empty and 'sentiment' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Distribution
    sns.histplot(df['sentiment'], bins=20, kde=True, ax=axes[0], color='#6C63FF')
    axes[0].set_title('Sentiment Distribution')
    axes[0].set_xlabel('Sentiment Score')
    axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)

    # Sentiment by source
    df.groupby('source')['sentiment'].mean().plot(kind='bar', ax=axes[1],
                                                   color=['#6C63FF', '#48CAE4'], edgecolor='none')
    axes[1].set_title('Avg Sentiment by Source')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()

    pos = (df['sentiment'] > 0).sum()
    neg = (df['sentiment'] < 0).sum()
    neu = (df['sentiment'] == 0).sum()
    print(f'Positive: {pos} | Negative: {neg} | Neutral: {neu}')

## Top Keywords

In [ ]:
if not df.empty and 'keyword' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    kw_counts = df['keyword'].value_counts().head(10)
    kw_counts.plot(kind='barh', ax=ax, color='#6C63FF', edgecolor='none')
    ax.set_title('Top 10 Keywords', fontsize=14)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## Word Cloud from Titles

In [ ]:
if not df.empty and 'title' in df.columns:
    text = ' '.join(df['title'].dropna().tolist())
    if text.strip():
        wc = WordCloud(width=900, height=400, background_color='#1e1e2e',
                       colormap='cool', max_words=100).generate(text)
        plt.figure(figsize=(12, 5))
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word Cloud — Post Titles', fontsize=14, color='white', pad=10)
        plt.tight_layout()
        plt.show()

## Posts Over Time

In [ ]:
if not df.empty and 'date' in df.columns:
    df_time = df.dropna(subset=['date'])
    if not df_time.empty:
        df_time = df_time.set_index('date').resample('D').size()
        fig, ax = plt.subplots(figsize=(12, 4))
        df_time.plot(ax=ax, color='#6C63FF', linewidth=2)
        ax.fill_between(df_time.index, df_time.values, alpha=0.2, color='#6C63FF')
        ax.set_title('Posts Over Time', fontsize=14)
        ax.set_xlabel('Date')
        ax.set_ylabel('Posts')
        plt.tight_layout()
        plt.show()
    else:
        print('No date data available')